In [ ]:
import SimpleITK as sitk
import numpy as np
import os

def extract_rectum_patch(original_image_path, prostate_mask_path, output_patch_path):
    """
    Calculate the bounding box of the rectum region based on the prostate mask
    and extract a patch from the original image, saving it as a single 3D image.

    Parameters:
    - original_image_path: Path to the original MR image.
    - prostate_mask_path: Path to the prostate.nii.gz mask.
    - output_patch_path: Path to save the extracted patch image.

    Returns:
    - None
    """
    # Load the prostate mask
    prostate_mask_image = sitk.ReadImage(prostate_mask_path)
    prostate_mask_array = sitk.GetArrayFromImage(prostate_mask_image)  # Shape: (Z, Y, X)

    image_size = prostate_mask_array.shape  # (Z, Y, X)

    # Calculate the area of the prostate mask on each slice
    areas = []
    for z in range(image_size[0]):  # Slices along the Z-axis
        slice_mask = prostate_mask_array[z, :, :]
        area = np.count_nonzero(slice_mask)
        areas.append(area)

    areas = np.array(areas)
    max_area_index = np.argmax(areas)

    # Extract the mask on the slice with the largest area
    largest_slice_mask = prostate_mask_array[max_area_index, :, :]

    # Find the centroid (x0, y0) of the mask on that slice
    indices = np.argwhere(largest_slice_mask)
    if len(indices) == 0:
        raise ValueError("No non-zero pixels found on the slice with the largest area.")

    y_coords, x_coords = indices[:, 0], indices[:, 1]  # Y, X coordinates

    x0 = int(round(x_coords.mean()))
    y0 = int(round(y_coords.mean()))

    ymin = y_coords.min()
    L0 = y0 - ymin

    if L0 <= 0:
        raise ValueError("Invalid L0 calculated. L0 should be a positive value.")

    # Estimate the center coordinates of the rectum
    rectum_center_y = int(round(y0 - 1.5 * L0))
    rectum_center_x = x0

    # Calculate the bounding box with a side length of 0.5 * L0
    side_length = int(round(0.5 * L0))
    if side_length < 1:
        side_length = 1  # Ensure the side length is at least 1

    # Calculate half of the side length
    half_side = side_length // 2

    # Calculate the coordinates of the bounding box
    x_min = rectum_center_x - half_side
    x_max = rectum_center_x + half_side - 1 if side_length % 2 == 0 else rectum_center_x + half_side
    y_min = rectum_center_y - half_side
    y_max = rectum_center_y + half_side - 1 if side_length % 2 == 0 else rectum_center_y + half_side

    # Ensure the coordinates are within the image bounds
    image_width = image_size[2]  # X dimension
    image_height = image_size[1]  # Y dimension

    if x_min < 0:
        x_max += -x_min
        x_min = 0
    if x_max > image_width - 1:
        x_min -= (x_max - (image_width - 1))
        x_max = image_width - 1
        if x_min < 0:
            x_min = 0  # Cannot adjust further

    if y_min < 0:
        y_max += -y_min
        y_min = 0
    if y_max > image_height - 1:
        y_min -= (y_max - (image_height - 1))
        y_max = image_height - 1
        if y_min < 0:
            y_min = 0

    # Ensure x_max >= x_min and y_max >= y_min
    if x_max < x_min or y_max < y_min:
        raise ValueError("Invalid bounding box coordinates.")

    # Load the original image
    original_image = sitk.ReadImage(original_image_path)
    original_array = sitk.GetArrayFromImage(original_image)  # Shape: (Z, Y, X)

    # Extract patches from each slice and collect them into a list
    patches = []
    for z in range(image_size[0]):
        slice_image = original_array[z, :, :]
        patch = slice_image[y_min:y_max + 1, x_min:x_max + 1]
        patches.append(patch)

    # Stack patches along the Z-axis to form a 3D array
    patch_array = np.stack(patches, axis=0)  # Shape: (Z, patch_height, patch_width)

    # Convert the patch array to a SimpleITK image
    patch_image = sitk.GetImageFromArray(patch_array)

    # Set the same pixel spacing and origin as the original image
    original_spacing = original_image.GetSpacing()  # (spacing_x, spacing_y, spacing_z)
    original_origin = original_image.GetOrigin()
    original_direction = original_image.GetDirection()
    # The spacing and direction need to be adjusted if axes are reordered
    patch_image.SetSpacing((original_spacing[0], original_spacing[1], original_spacing[2]))
    patch_image.SetOrigin(original_origin)
    patch_image.SetDirection(original_direction)

    # Save the patch image
    sitk.WriteImage(patch_image, output_patch_path)
    print(f"Saved extracted patch to {output_patch_path}")

# Example usage:

original_image_path = r'/path/to/workspace/classification_model_originalimage/train_data/SUBJECT_ID_REDACTED_4_ax_t2_frfse_ardl_h_1nex.nii.gz'  
prostate_mask_path = r'/path/to/workspace/classification_model_originalimage/totalsegment_work/train/SUBJECT_ID_REDACTED_4_ax_t2_frfse_ardl_h_1nex/prostate.nii.gz'   
output_directory = r'/path/to/workspace/classification_model_originalimage/'                # Replace with your desired output directory
patch_name = 'extracted_patch3.nii.gz'
output_patch_path = os.path.join(output_directory, patch_name)

extract_rectum_patch(original_image_path, prostate_mask_path, output_patch_path)
print('done')
